# DeepONet Go AI — Training Notebook
Run cells top to bottom. After Cell 5 starts, you can close the tab (Colab Pro background execution).

In [ ]:
# ── Cell 1: Mount Drive + Clone Repo ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/OperatorGo/checkpoints', exist_ok=True)

!git clone https://github.com/nuralik/OperatorGo-.git
%cd OperatorGo-

# Checkpoints go to Drive — survive disconnects
!rm -rf checkpoints
!ln -s /content/drive/MyDrive/OperatorGo/checkpoints checkpoints
print('Done. Checkpoints linked to Google Drive.')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────
!pip install sgfmill beautifulsoup4 -q
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 3: Download 9x9 SGF Training Data ────────────────────────────────
import os, requests, time
from pathlib import Path
from bs4 import BeautifulSoup

BASE = 'https://homepages.cwi.nl/~aeb/go/games/games/other_sizes/9x9'
SUBDIRS = [
    'Go_Seigen', 'Minigo', 'Misc', 'ProPairgo', 'computer',
    'NHK/NewYear1990', 'NHK/NewYear2002', 'NHK/NewYear2003'
]

total = 0
for subdir in SUBDIRS:
    out_dir = Path(f'data/sgf_raw/cwi_9x9/{subdir}')
    out_dir.mkdir(parents=True, exist_ok=True)

    r = requests.get(f'{BASE}/{subdir}/', timeout=15)
    soup = BeautifulSoup(r.text, 'html.parser')
    files = [a['href'] for a in soup.find_all('a', href=True)
             if a['href'].endswith('.sgf')]

    for f in files:
        dest = out_dir / f
        if not dest.exists():  # skip if already downloaded
            r2 = requests.get(f'{BASE}/{subdir}/{f}', timeout=15)
            dest.write_bytes(r2.content)
            total += 1

    print(f'  {subdir}: {len(files)} files')

all_sgf = list(Path('data/sgf_raw').rglob('*.sgf'))
print(f'\nTotal SGF files: {len(all_sgf)}')

In [ ]:
# ── Cell 4: Supervised Training (DeepONet, latent_dim=256) ────────────────
# Skip this cell if checkpoints/best_deeponet.pt already exists on Drive
import os
if os.path.exists('checkpoints/best_deeponet.pt'):
    print('Supervised checkpoint found — skipping supervised training.')
else:
    !python -m training.train \
        --model deeponet \
        --data data/sgf_raw/cwi_9x9 \
        --epochs 30 \
        --batch_size 256 \
        --filters 64 \
        --res_blocks 5 \
        --latent_dim 256 \
        --save_dir checkpoints

In [ ]:
# ── Cell 5: Self-Play Training ─────────────────────────────────────────────
# Runs until max_iters or you stop it. Checkpoint saved every iteration.
# Safe to re-run — automatically resumes if checkpoint exists.
import os

resume_flag = '--resume' if os.path.exists('checkpoints/self_play_latest.pt') else ''
print(f'Starting self-play  {"(resuming)" if resume_flag else "(fresh start)"}')

!python -m training.self_play_train \
    {resume_flag} \
    --simulations 100 \
    --games_per_iter 20 \
    --train_steps 200 \
    --workers 4 \
    --latent_dim 256 \
    --max_iters 2000

In [ ]:
# ── Cell 6: Check Training Progress ───────────────────────────────────────
# Run this anytime to see what checkpoints exist
import os, torch

for fname in ['best_deeponet.pt', 'self_play_latest.pt', 'self_play_best.pt']:
    path = f'checkpoints/{fname}'
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        iteration = ckpt.get('iteration', 'supervised')
        val_loss  = ckpt.get('val_loss', ckpt.get('best_loss', 'n/a'))
        size_mb   = os.path.getsize(path) / 1e6
        print(f'{fname:<30} iter={iteration}  loss={val_loss}  ({size_mb:.1f} MB)')
    else:
        print(f'{fname:<30} not found')